# KubeMedic GRPO Training

Self-contained version of the KubeMedic single-session GRPO trainer.

Run cells top to bottom. Designed for a GPU runtime (Colab T4/L4/A100, HF Spaces with GPU, etc.). On CPU the smoke test still works but training will be slow / unsupported by 4-bit quantization.

## 1. Install dependencies

Pulls the KubeMedic env client (`openenv-kubemedic`) plus the full training stack: TRL + vLLM, transformers, accelerate, peft, datasets, bitsandbytes, plotting, and wandb.

In [2]:
import os, shutil, subprocess

SPACE_ID = 'ashiqabdulkhader/Kubemedic'
SPACE_URL = f'https://huggingface.co/spaces/{SPACE_ID}.git'
KUBEMEDIC_SRC = '/tmp/kubemedic-src'

if os.path.isdir(KUBEMEDIC_SRC):
    shutil.rmtree(KUBEMEDIC_SRC)
print('Cloning Space:', SPACE_URL)
subprocess.check_call(['git', 'clone', '--depth', '1', SPACE_URL, KUBEMEDIC_SRC])

%pip install -q -U "numpy>=2.0,<3"
%pip install -q -U "{KUBEMEDIC_SRC}[train]" "wandb>=0.17.0"

print('\nInstall complete. If you see numpy ABI errors later, restart the kernel once and re-run.')

Cloning Space: https://huggingface.co/spaces/ashiqabdulkhader/Kubemedic.git


Cloning into '/tmp/kubemedic-src'...


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.

*** IMPORTANT ***
After this cell finishes, RESTART the runtime once (Runtime > Restart runtime),
then run cells from the top again. This avoids numpy/pyarrow ABI mismatches.


## 2. Imports and environment setup

Loads `HF_TOKEN` (from `.env`, Colab `userdata`, or the `hf` CLI), sets matplotlib to a headless backend, and imports the training stack.

In [1]:
import argparse
import csv
import json
import math
import os
import random
import shlex
import subprocess
import sys
import tempfile
import time
from pathlib import Path
from typing import Any

os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'kubemedic-mpl'))
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('TRL_EXPERIMENTAL_SILENCE', '1')
os.environ.setdefault('WANDB_DISABLED', 'true')

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
env_path = next((path for path in env_candidates if path.exists()), None)
if env_path:
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)

if not os.environ.get('HF_TOKEN'):
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        pass

if not os.environ.get('HF_TOKEN'):
    try:
        os.environ['HF_TOKEN'] = subprocess.check_output(['hf', 'auth', 'token'], text=True).strip()
    except Exception:
        pass

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import transformers
import wandb
from datasets import Dataset
from packaging.version import Version
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import GRPOConfig, GRPOTrainer
from trl.experimental.openenv import generate_rollout_completions

from Kubemedic import KubemedicAction, KubemedicEnv, KubemedicObservation
from Kubemedic.models import ToolName

try:
    from websockets.exceptions import ConnectionClosedError, ConnectionClosedOK
except Exception:
    ConnectionClosedError = Exception
    ConnectionClosedOK = Exception

print('Kernel python   :', sys.executable)
print('Torch version   :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
print('HF token loaded :', bool(os.environ.get('HF_TOKEN')))
print('WANDB key loaded:', bool(os.environ.get('WANDB_API_KEY')))

Hint: Run `hf auth whoami` to see which account this token belongs to.
/home/ubuntu/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 2.2.6
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 3. Configuration

Hyperparameters, scenario list, system prompt, output paths. Edit any of these knobs in this single cell.

In [ ]:
ENV_URL = f"https://{SPACE_ID.replace('/', '-')}.hf.space"
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
OUTPUT_DIR = Path('outputs/kubemedic-qwen25-3b-grpo')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_EPISODES = 8
EVAL_EPISODES = 2
NUM_GENERATIONS = 2
PER_DEVICE_TRAIN_BATCH_SIZE = NUM_GENERATIONS
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 1e-5
MAX_COMPLETION_LENGTH = 256
MAX_TURNS = 8
SAVE_STEPS = 4
LOGGING_STEPS = 1
SEED = 42
VLLM_MODE = 'colocate'
VLLM_GPU_MEMORY_UTILIZATION = 0.35
USE_4BIT = True
USE_VLLM = True
SKIP_SMOKE_TEST = False
SMOKE_ONLY = False
WANDB_PROJECT = 'kubemedic-grpo'
WANDB_RUN_NAME = None
DEFAULT_SCENARIOS = ['KUBE-01', 'KUBE-03', 'KUBE-04', 'KUBE-05', 'KUBE-06']
SCENARIOS = list(DEFAULT_SCENARIOS)

CONNECT_TIMEOUT_S = 30.0
MESSAGE_TIMEOUT_S = 240.0
VALID_TOOLS = set(ToolName.__args__)  # type: ignore[attr-defined]

SYSTEM_PROMPT = """You are a Kubernetes SRE agent operating KubeMedic.
Diagnose the cluster carefully, use the available actions, minimize blast radius, and stop once the cluster is healthy.

Respond with exactly one action per turn using this format:
- TOOL_NAME key=value key=value
- TOOL_NAME {\"json\": \"object\"}

Allowed tool names:
kubectl_get
kubectl_describe
kubectl_logs
kubectl_top_pods
kubectl_top_nodes
kubectl_patch_resources
kubectl_patch_tolerations
kubectl_cordon
kubectl_uncordon
kubectl_delete_pod
kubectl_delete_workload

Rules:
- Never include markdown fences or explanations.
- Prefer inspect-first behavior before mutating resources.
- Use namespace=challenge unless the observation clearly requires something else.
"""

TRAINING_HINT = (
    'Diagnose and repair this Kubernetes incident. Output one action only, in the required syntax.'
)

print('ENV_URL   :', ENV_URL)
print('MODEL_ID  :', MODEL_ID)
print('OUTPUT_DIR:', OUTPUT_DIR.resolve())
print('SCENARIOS :', SCENARIOS)

## 4. Helper functions

Observation formatting, action parsing, prompt rendering, retry/connection helpers, and rollout utilities. These are the pure-Python helpers from the script.

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def configure_wandb(project: str, run_name: str | None) -> list[str]:
    api_key = os.environ.get('WANDB_API_KEY', '').strip()
    if not api_key:
        os.environ['WANDB_DISABLED'] = 'true'
        print('WANDB_API_KEY not set; wandb disabled.')
        return []
    os.environ['WANDB_DISABLED'] = 'false'
    os.environ.setdefault('WANDB_PROJECT', project)
    if run_name:
        os.environ.setdefault('WANDB_NAME', run_name)
    try:
        import wandb
        wandb.login(key=api_key, relogin=True)
        wandb_project = os.environ['WANDB_PROJECT']
        wandb_run = os.environ.get('WANDB_NAME')
        msg = f'wandb authenticated; project={wandb_project}'
        if wandb_run:
            msg += f' run={wandb_run}'
        print(msg)
    except Exception as exc:
        print('wandb.login failed:', exc, '-- disabling wandb for this run.')
        os.environ['WANDB_DISABLED'] = 'true'
        return []
    return ['wandb']


def make_env(env_url: str):
    return KubemedicEnv(
        base_url=env_url,
        connect_timeout_s=CONNECT_TIMEOUT_S,
        message_timeout_s=MESSAGE_TIMEOUT_S,
    ).sync()


def format_observation(obs: KubemedicObservation) -> str:
    pods = []
    for pod in obs.pods:
        parts = [f'phase={pod.phase}']
        if pod.reason:
            parts.append(f'reason={pod.reason}')
        if pod.restarts:
            parts.append(f'restarts={pod.restarts}')
        pods.append(f'- {pod.name} ({pod.namespace}): ' + ' '.join(parts))

    nodes = []
    for node in obs.nodes:
        pressure = [c.type for c in node.conditions if c.status == 'True' and c.type.endswith('Pressure')]
        nodes.append(
            f"- {node.name}: ready={node.ready} pressure={','.join(pressure) if pressure else 'none'}"
        )

    info = obs.info or {}
    reward_breakdown = info.get('reward_breakdown', {})
    breakdown_text = ', '.join(f'{k}={v:.2f}' for k, v in reward_breakdown.items()) if reward_breakdown else 'none'

    return (
        f'{TRAINING_HINT}\n\n'
        f'Scenario: {obs.scenario}\n'
        f'Time step: {obs.t}\n'
        f"Blocked reason: {obs.blocked_reason or 'none'}\n"
        f'Last reward: {float(obs.reward or 0.0):.2f}\n'
        f"Disruptions: {info.get('disruptions', 0)}\n"
        f'Reward breakdown: {breakdown_text}\n\n'
        f"Pods:\n{chr(10).join(pods) if pods else '- none'}\n\n"
        f"Nodes:\n{chr(10).join(nodes) if nodes else '- none'}"
    )


def format_history(history: list[dict[str, Any]]) -> str:
    if not history:
        return ''
    lines = ['Previous actions:']
    for item in history:
        output = item['output']
        if len(output) > 220:
            output = output[:220] + '...'
        lines.append(f"- {item['action']} -> reward={item['reward']:.2f} | {output}")
    return '\n'.join(lines)


def coerce_value(value: str) -> Any:
    lowered = value.lower()
    if lowered == 'true':
        return True
    if lowered == 'false':
        return False
    if lowered in ('none', 'null'):
        return None
    try:
        return json.loads(value)
    except Exception:
        pass
    try:
        return int(value)
    except ValueError:
        pass
    return value


def parse_action_text(text: str) -> KubemedicAction | None:
    cleaned = text.strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.strip('`').strip()
    line = cleaned.splitlines()[0].strip() if cleaned else ''
    if not line:
        return None

    parts = line.split(maxsplit=1)
    tool = parts[0].strip()
    if tool not in VALID_TOOLS:
        return None

    if len(parts) == 1:
        return KubemedicAction(tool=tool, args={})

    remainder = parts[1].strip()
    if not remainder:
        return KubemedicAction(tool=tool, args={})

    args: dict[str, Any] = {}
    if remainder.startswith('{'):
        parsed = json.loads(remainder)
        if not isinstance(parsed, dict):
            return None
        args = parsed
    else:
        for token in shlex.split(remainder):
            if '=' not in token:
                continue
            key, value = token.split('=', 1)
            args[key] = coerce_value(value)

    return KubemedicAction(tool=tool, args=args)


def render_prompt(tokenizer, messages: list[dict[str, str]]) -> str:
    if getattr(tokenizer, 'chat_template', None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    system_lines = [msg['content'] for msg in messages if msg['role'] == 'system']
    user_lines = [msg['content'] for msg in messages if msg['role'] == 'user']
    return '\n\n'.join(
        [
            *(['SYSTEM:\n' + '\n'.join(system_lines)] if system_lines else []),
            *(['USER:\n' + '\n'.join(user_lines)] if user_lines else []),
            'ASSISTANT:\n',
        ]
    )


def is_connection_error(exc: Exception) -> bool:
    text = str(exc)
    return (
        isinstance(exc, (ConnectionClosedError, ConnectionClosedOK))
        or 'ConnectionClosed' in exc.__class__.__name__
        or 'no close frame received or sent' in text
        or 'close frame' in text
        or 'websocket' in text.lower()
    )


def build_dataset(num_rows: int) -> Dataset:
    rows = [{'prompt': TRAINING_HINT} for _ in range(num_rows)]
    return Dataset.from_list(rows)


def infer_lora_target_modules(model) -> list[str]:
    model_type = getattr(model.config, 'model_type', '')
    if model_type in {'qwen2', 'qwen2_5', 'qwen3'}:
        return ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    if model_type in {'gpt2'}:
        return ['c_attn', 'c_proj', 'c_fc']
    module_names = {name.split('.')[-1] for name, _ in model.named_modules()}
    preferred = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj', 'c_attn', 'c_proj', 'c_fc']
    inferred = [name for name in preferred if name in module_names]
    if inferred:
        return inferred
    raise ValueError(f'Could not infer LoRA target modules for model_type={model_type!r}.')


def reward_total(completions: list[str], **kwargs) -> list[float]:
    rewards = kwargs.get('total_reward') if kwargs else None
    return [float(value) for value in rewards] if rewards else [0.0 for _ in completions]

print('Helpers ready.')

## 5. Rollout utilities

Local generation fallback (used when vLLM is off) and the per-episode rollout that drives the live KubeMedic environment.

In [ ]:
def generate_rollout_completions_local(trainer: GRPOTrainer, prompts: list[str]) -> list[dict[str, Any]]:
    tokenizer = trainer.processing_class
    model = trainer.model
    results: list[dict[str, Any]] = []

    for prompt_text in prompts:
        encoded = tokenizer(prompt_text, return_tensors='pt')
        encoded = {key: value.to(model.device) for key, value in encoded.items()}
        generation = model.generate(
            **encoded,
            do_sample=True,
            temperature=trainer.temperature,
            top_p=trainer.top_p if trainer.top_p is not None else 1.0,
            top_k=trainer.top_k if trainer.top_k is not None else 50,
            max_new_tokens=trainer.max_completion_length,
            return_dict_in_generate=True,
            output_scores=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        sequence = generation.sequences[0]
        prompt_len = encoded['input_ids'].shape[1]
        completion_ids = sequence[prompt_len:]
        if generation.scores:
            transition_scores = model.compute_transition_scores(
                generation.sequences,
                generation.scores,
                normalize_logits=True,
            )[0]
            logprobs = [float(score) for score in transition_scores.tolist()[: len(completion_ids)]]
        else:
            logprobs = []

        results.append(
            {
                'prompt_ids': sequence[:prompt_len].tolist(),
                'completion_ids': completion_ids.tolist(),
                'logprobs': logprobs,
                'text': tokenizer.decode(completion_ids, skip_special_tokens=True),
            }
        )
    return results


def rollout_once(
    trainer: GRPOTrainer,
    env_url: str,
    tokenizer,
    scenario: str,
    max_turns: int,
    max_total_tokens: int,
    max_retries: int = 3,
) -> dict[str, Any]:
    last_exc: Exception | None = None
    for attempt in range(1, max_retries + 1):
        env = make_env(env_url)
        try:
            result = env.reset(scenario=scenario)
            observation = result.observation

            prompt_ids: list[int] = []
            completion_ids: list[int] = []
            logprobs: list[float] = []
            step_rewards: list[float] = []
            history: list[dict[str, Any]] = []

            for _ in range(max_turns):
                if result.done or len(completion_ids) >= max_total_tokens:
                    break

                user_parts = [format_observation(observation)]
                history_text = format_history(history)
                if history_text:
                    user_parts.append(history_text)
                messages = [
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': '\n\n'.join(user_parts)},
                ]

                prompt_text = render_prompt(tokenizer, messages)
                if trainer.use_vllm:
                    rollout_output = generate_rollout_completions(trainer, [prompt_text])[0]
                else:
                    rollout_output = generate_rollout_completions_local(trainer, [prompt_text])[0]
                prompt_ids.extend(list(rollout_output['prompt_ids']))
                completion_ids.extend(list(rollout_output['completion_ids']))
                logprobs.extend([float(v) for v in rollout_output['logprobs']])

                completion_text = (rollout_output.get('text') or '').strip()
                action = parse_action_text(completion_text)
                if action is None:
                    step_rewards.append(-1.0)
                    history.append(
                        {
                            'action': completion_text or '(empty)',
                            'output': 'Invalid action format.',
                            'reward': -1.0,
                        }
                    )
                    continue

                result = env.step(action)
                observation = result.observation
                reward = float(result.reward or 0.0)
                step_rewards.append(reward)
                tool_output = json.dumps(observation.tool_result or {}, default=str)
                history.append({'action': completion_text, 'output': tool_output, 'reward': reward})

            total_reward = sum(step_rewards) if step_rewards else -1.0
            return {
                'prompt_ids': prompt_ids,
                'completion_ids': completion_ids,
                'logprobs': logprobs,
                'total_reward': total_reward,
                'steps': len(step_rewards),
                'scenario': scenario,
                'resolved': bool(result.done),
            }
        except Exception as exc:
            last_exc = exc
            if not is_connection_error(exc) or attempt >= max_retries:
                raise
            wait_s = min(2 ** (attempt - 1), 8)
            print(
                f'Episode websocket dropped during scenario {scenario} '
                f'(attempt {attempt}/{max_retries}): {exc}. Retrying in {wait_s}s...'
            )
            time.sleep(wait_s)
        finally:
            try:
                env.close()
            except Exception:
                pass

    raise RuntimeError(f'Episode failed after {max_retries} reconnect attempts: {last_exc}')

print('Rollout helpers ready.')

## 6. Smoke test the deployed Space

Hits the live KubeMedic environment to confirm the Space is reachable and tool calls return observations. Skip this cell by setting `SKIP_SMOKE_TEST = True` in the config cell.

In [ ]:
def smoke_test(env_url: str, scenario: str) -> None:
    env = make_env(env_url)
    try:
        result = env.reset(scenario=scenario)
        print(format_observation(result.observation)[:1200])
        step_result = env.step(KubemedicAction(tool='kubectl_get', args={'resource': 'pods', 'namespace': 'challenge'}))
        print('\n--- smoke tool call ---\n')
        print(format_observation(step_result.observation)[:1200])
        print('\nSmoke test passed.')
    finally:
        env.close()

if not SKIP_SMOKE_TEST:
    smoke_test(ENV_URL, SCENARIOS[0])
else:
    print('Skipping smoke test.')

## 7. Tokenizer and base model

Loads Qwen2.5-3B-Instruct with optional 4-bit quantization (when CUDA + bitsandbytes are available). Sets seeds and validates the transformers version.

In [ ]:
if SMOKE_ONLY:
    raise SystemExit('SMOKE_ONLY=True; stop here.')

if PER_DEVICE_TRAIN_BATCH_SIZE % NUM_GENERATIONS != 0:
    raise ValueError('PER_DEVICE_TRAIN_BATCH_SIZE must be divisible by NUM_GENERATIONS for GRPO.')

if Version(transformers.__version__) < Version('4.56.0'):
    raise RuntimeError('transformers>=4.56.0 is required for the training entrypoint.')

set_seed(SEED)
report_to = configure_wandb(WANDB_PROJECT, WANDB_RUN_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_vllm = torch.cuda.is_available() and USE_VLLM
if torch.cuda.is_available():
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    dtype = torch.float32

quant_config = None
if torch.cuda.is_available() and USE_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=dtype,
        bnb_4bit_use_double_quant=True,
    )

model_kwargs: dict[str, Any] = {
    'torch_dtype': dtype,
    'quantization_config': quant_config,
}
if torch.cuda.is_available():
    model_kwargs['device_map'] = 'auto'

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=infer_lora_target_modules(model),
)

print('use_vllm  :', use_vllm)
print('dtype     :', dtype)
print('4-bit     :', quant_config is not None)
print('LoRA tgts :', peft_config.target_modules)

## 8. GRPO trainer setup and training

Builds the train/eval datasets, the GRPO config, the rollout function (which logs episodes to disk), and runs `trainer.train()`.

In [ ]:
train_dataset = build_dataset(TRAIN_EPISODES)
eval_dataset = build_dataset(EVAL_EPISODES)

grpo_config = GRPOConfig(
    output_dir=str(OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    num_train_epochs=1,
    save_steps=SAVE_STEPS,
    logging_steps=LOGGING_STEPS,
    save_total_limit=2,
    bf16=(dtype == torch.bfloat16),
    fp16=(dtype == torch.float16),
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to=report_to,
    use_vllm=use_vllm,
    vllm_mode=VLLM_MODE,
    vllm_gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
    log_completions=False,
    max_tool_calling_iterations=1,
    seed=SEED,
)

reward_log_path = OUTPUT_DIR / 'episode_rewards.csv'
transcript_path = OUTPUT_DIR / 'agent_transcripts.jsonl'
with reward_log_path.open('w', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['episode', 'scenario', 'total_reward', 'steps', 'resolved'])

if 'wandb' in report_to and wandb.run is None:
    wandb.init(
        project=os.environ.get('WANDB_PROJECT', WANDB_PROJECT),
        name=os.environ.get('WANDB_NAME', WANDB_RUN_NAME),
        config={
            'model_id': MODEL_ID,
            'env_url': ENV_URL,
            'train_episodes': TRAIN_EPISODES,
            'eval_episodes': EVAL_EPISODES,
            'num_generations': NUM_GENERATIONS,
            'per_device_train_batch_size': PER_DEVICE_TRAIN_BATCH_SIZE,
            'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
            'learning_rate': LEARNING_RATE,
            'max_completion_length': MAX_COMPLETION_LENGTH,
            'max_turns': MAX_TURNS,
            'use_vllm': use_vllm,
            'vllm_mode': VLLM_MODE,
            'use_4bit': USE_4BIT,
            'seed': SEED,
            'scenarios': SCENARIOS,
        },
        tags=['kubemedic', 'grpo', 'qwen2.5'],
        reinit=True,
    )
    print('wandb run:', wandb.run.url if wandb.run else '(none)')

episode_counter = {'value': 0}
episode_table = wandb.Table(columns=['episode', 'scenario', 'total_reward', 'steps', 'resolved']) if wandb.run else None

def rollout_func(prompts: list[str], trainer: GRPOTrainer) -> dict[str, list]:
    prompt_count = len(prompts)
    episode_prompt_ids: list[list[int]] = []
    episode_completion_ids: list[list[int]] = []
    episode_logprobs: list[list[float]] = []
    total_rewards: list[float] = []
    steps: list[int] = []
    scenario_names: list[str] = []
    resolved_flags: list[bool] = []

    for start in range(0, prompt_count, NUM_GENERATIONS):
        scenario = random.choice(SCENARIOS)
        group_size = min(NUM_GENERATIONS, prompt_count - start)
        for _ in range(group_size):
            episode = rollout_once(
                trainer=trainer,
                env_url=ENV_URL,
                tokenizer=tokenizer,
                scenario=scenario,
                max_turns=MAX_TURNS,
                max_total_tokens=4096,
            )
            episode_prompt_ids.append(episode['prompt_ids'])
            episode_completion_ids.append(episode['completion_ids'])
            episode_logprobs.append(episode['logprobs'])
            total_rewards.append(float(episode['total_reward']))
            steps.append(int(episode['steps']))
            scenario_names.append(episode['scenario'])
            resolved_flags.append(bool(episode['resolved']))

            episode_counter['value'] += 1
            with reward_log_path.open('a', newline='') as handle:
                writer = csv.writer(handle)
                writer.writerow([
                    episode_counter['value'],
                    episode['scenario'],
                    episode['total_reward'],
                    episode['steps'],
                    episode['resolved'],
                ])
            with transcript_path.open('a') as handle:
                handle.write(json.dumps(episode) + '\n')

            if wandb.run is not None:
                scenario_index = SCENARIOS.index(episode['scenario']) if episode['scenario'] in SCENARIOS else -1
                wandb.log(
                    {
                        'rollout/episode': episode_counter['value'],
                        'rollout/total_reward': float(episode['total_reward']),
                        'rollout/steps': int(episode['steps']),
                        'rollout/resolved': int(bool(episode['resolved'])),
                        'rollout/scenario_index': scenario_index,
                    }
                )
                if episode_table is not None:
                    episode_table.add_data(
                        episode_counter['value'],
                        episode['scenario'],
                        float(episode['total_reward']),
                        int(episode['steps']),
                        bool(episode['resolved']),
                    )

    return {
        'prompt_ids': episode_prompt_ids,
        'completion_ids': episode_completion_ids,
        'logprobs': episode_logprobs,
        'total_reward': total_rewards,
        'steps': steps,
        'scenario': scenario_names,
        'resolved': resolved_flags,
    }

os.environ.setdefault('VLLM_WORKER_MULTIPROC_METHOD', 'spawn')

import sys as _sys
_nb_stdout, _nb_stderr = _sys.stdout, _sys.stderr
if hasattr(_sys.__stdout__, 'fileno') and hasattr(_sys.__stderr__, 'fileno'):
    _sys.stdout, _sys.stderr = _sys.__stdout__, _sys.__stderr__
try:
    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=reward_total,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=grpo_config,
        rollout_func=rollout_func,
        peft_config=peft_config,
    )
finally:
    _sys.stdout, _sys.stderr = _nb_stdout, _nb_stderr

trainer.train()

## 9. Save artifacts and plots

Saves the LoRA adapter + tokenizer, writes the reward CSV plot, the trainer log history, and a JSON training summary.

In [ ]:
def plot_rewards(csv_path: Path, out_path: Path) -> None:
    rows = pd.read_csv(csv_path)
    if rows.empty:
        return
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(rows['episode'], rows['total_reward'], marker='o', alpha=0.35, label='episode reward')
    rolling = rows['total_reward'].rolling(window=min(10, len(rows)), min_periods=1).mean()
    ax.plot(rows['episode'], rolling, linewidth=2, label='rolling mean')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Reward')
    ax.set_title('KubeMedic Single-Session GRPO Reward Curve')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close(fig)


def plot_trainer_metrics(log_history: pd.DataFrame, out_path: Path) -> None:
    if log_history.empty:
        return
    cols = [name for name in ['loss', 'grad_norm', 'learning_rate'] if name in log_history.columns]
    if not cols:
        return
    fig, axes = plt.subplots(1, len(cols), figsize=(6 * len(cols), 4))
    if len(cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, cols):
        subset = log_history.dropna(subset=[col])
        if subset.empty or 'step' not in subset.columns:
            continue
        sns.lineplot(data=subset, x='step', y=col, ax=ax)
        ax.set_title(col)
    plt.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close(fig)

adapter_dir = OUTPUT_DIR / 'adapter'
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

reward_plot_path = OUTPUT_DIR / 'reward_plot.png'
trainer_metrics_path = OUTPUT_DIR / 'trainer_metrics.png'
log_history_path = OUTPUT_DIR / 'trainer_log_history.csv'
summary_path = OUTPUT_DIR / 'training_summary.json'

plot_rewards(reward_log_path, reward_plot_path)
log_history = pd.DataFrame(trainer.state.log_history)
log_history.to_csv(log_history_path, index=False)
plot_trainer_metrics(log_history, trainer_metrics_path)

summary = {
    'model_id': MODEL_ID,
    'env_url': ENV_URL,
    'train_episodes': TRAIN_EPISODES,
    'eval_episodes': EVAL_EPISODES,
    'num_generations': NUM_GENERATIONS,
    'output_dir': str(OUTPUT_DIR),
    'adapter_dir': str(adapter_dir),
    'wandb_run_url': wandb.run.url if wandb.run else None,
}
summary_path.write_text(json.dumps(summary, indent=2))
print('Wrote artifacts under', OUTPUT_DIR.resolve())

if wandb.run is not None:
    if reward_plot_path.exists():
        wandb.log({'plots/reward_curve': wandb.Image(str(reward_plot_path))})
    if trainer_metrics_path.exists():
        wandb.log({'plots/trainer_metrics': wandb.Image(str(trainer_metrics_path))})
    if episode_table is not None:
        wandb.log({'rollout/episode_table': episode_table})

    run_artifact = wandb.Artifact('kubemedic-grpo-run', type='training-output')
    for path in (reward_log_path, log_history_path, summary_path,
                 reward_plot_path, trainer_metrics_path, transcript_path):
        if path.exists():
            run_artifact.add_file(str(path))
    wandb.log_artifact(run_artifact)

    if adapter_dir.exists():
        adapter_artifact = wandb.Artifact('kubemedic-grpo-adapter', type='model',
                                          metadata={'base_model': MODEL_ID, 'kind': 'lora'})
        adapter_artifact.add_dir(str(adapter_dir))
        wandb.log_artifact(adapter_artifact)

    wandb.finish()
    print('wandb run finished and artifacts uploaded.')
else:
    print('wandb not active; skipping wandb artifact upload.')

## 10. Inspect results

Read back the reward CSV, summary JSON, and plots saved above.

In [ ]:
from IPython.display import Image, display

reward_csv = OUTPUT_DIR / 'episode_rewards.csv'
summary_json = OUTPUT_DIR / 'training_summary.json'
reward_plot = OUTPUT_DIR / 'reward_plot.png'
trainer_plot = OUTPUT_DIR / 'trainer_metrics.png'

if reward_csv.exists():
    display(pd.read_csv(reward_csv).tail(10))
else:
    raise FileNotFoundError(
        f'{reward_csv} was not created. Check the training cell output above for the real failure.'
    )

if summary_json.exists():
    print(json.loads(summary_json.read_text()))
if reward_plot.exists():
    display(Image(filename=str(reward_plot)))
if trainer_plot.exists():
    display(Image(filename=str(trainer_plot)))